<a href="https://colab.research.google.com/github/EmIbrahimovic/sp26_6_4110_hw_colabs/blob/main/HW04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 4

## Imports and Utilities
**Note**: these imports and functions are available in catsoop. You do not need to copy them in.

In [1]:

from typing import (Callable, Iterable, List, Sequence, Tuple, Dict, Optional,
                    Any, Union)

from abc import abstractmethod
import collections
import itertools
import functools
import random

import numpy as np

########## Graph-Search-Related Utilities and Class Definitions ##########

State = Any
Action = Any

StateSeq = List[State]
ActionSeq = List[State]
CostSeq = RewardSeq = List[float]


class Problem(object):
  """The abstract base class for either a path cost problem or a reward problem."""

  def __init__(self, initial: State):
    self.initial = initial

  @abstractmethod
  def actions(self, state: State) -> Iterable[Action]:
    """Returns the allowed actions in a given state.

    The result would typically be a list. But if there are many actions,
    consider yielding them one at a time in an iterator,
    rather than building them all at once.
    """
    ...

  @abstractmethod
  def step(self, state: State, action: Action) -> State:
    """Returns the next state when executing a given action in a given state.

    The action must be one of self.actions(state).
    """
    ...


class PathCostProblem(Problem):
  """An abstract class for a path cost problem, based on AIMA.

  To formalize a path cost problem, you should subclass from this and implement
  the abstract methods.
  Then you will create instances of your subclass and solve them with the
  various search functions.
  """

  @abstractmethod
  def goal_test(self, state: State) -> bool:
    """Checks if the state is a goal."""
    ...

  @abstractmethod
  def step_cost(self, state1: State, action: Action, state2: State) -> float:
    """Returns the cost incurred at state2 from state1 via action."""
    ...

  def h(self, state: State) -> float:
    """Returns the heuristic value, a lower bound on the distance to goal."""
    return 0




## Best-first Search


### Utilities


**Note**: these imports and functions are available in catsoop. You do not need to copy them in.

In [2]:


class GridProblem(PathCostProblem):
  """A grid problem."""

  def __init__(self, initial=(0, 0), goal=(4, 4)):
    super().__init__(initial)
    self.goal = goal
    self.all_grid_actions = ["up", "down", "left", "right"]
    self.grid_act_to_delta = {
        "up": (-1, 0),
        "down": (1, 0),
        "left": (0, -1),
        "right": (0, 1)
    }
    # Somewhat unusual cost structure, depends on s', which is determined by s,a
    self.grid_arrival_costs = np.array(
        [
            [1, 1, 8, 1, 1],
            [1, 8, 1, 1, 1],
            [1, 8, 1, 1, 1],
            [1, 1, 1, 8, 1],
            [1, 1, 2, 1, 1],
        ],
        dtype=int,
    )

  def actions(self, state):
    (r, c) = state
    actions = []
    for act in self.all_grid_actions:
      dr, dc = self.grid_act_to_delta[act]
      new_r, new_c = r + dr, c + dc
      # Check if in bounds
      if (0 <= new_r < self.grid_arrival_costs.shape[0] and
          0 <= new_c < self.grid_arrival_costs.shape[1]):
        actions.append(act)
    return actions

  def step(self, state, action):
    (r, c) = state
    dr, dc = self.grid_act_to_delta[action]
    return (r + dr, c + dc)

  def goal_test(self, state):
    return state == self.goal

  def step_cost(self, state1, action, state2):
    return self.grid_arrival_costs[state2]

  def h(self, state):
    """Manhattan distance."""
    return abs(state[0] - self.goal[0]) + abs(state[1] - self.goal[1])


import contextlib


@contextlib.contextmanager
def count_method_calls(problem: Problem, *meths: str):
  """Track number of method invocations to a problem.

  Args:
    problem: an instance of a Problem.
    meths: a sequence of names for the methods to track call counts for.

  Example:
    >>> problem = GridProblem()
    >>> with count_method_calls(problem, "step") as counters:
    ...   problem.step((0, 0), "down")
    ...   problem.step((1, 1), "up")
    ...   assert counters["step"][((0, 0), "down")] == counters["step"][((1, 1), "up")]  == 1
    ...   assert counters["step"]["total"] == 2
  """
  counters = {meth: collections.Counter() for meth in meths}

  orig_problem_methods = {meth: getattr(problem, meth) for meth in meths}

  def meth_helper(attr, *args):
    counters[attr][args] += 1
    counters[attr]["total"] += 1
    return orig_problem_methods[attr](*args)

  for meth in meths:
    orig_problem_methods[meth] = getattr(problem, meth)
    setattr(problem, meth, functools.partial(meth_helper, meth))

  try:
    yield counters
  finally:
    for meth in meths:
      setattr(problem, meth, orig_problem_methods[meth])


**Note**: these imports and functions are available in catsoop. You do not need to copy them in.

In [3]:

import heapq as hq  # Can use this as a priority queue

# A useful data structure for best-first search
Node = collections.namedtuple("Node",
                              ["state", "parent", "action", "cost", "g"])


class SearchFailed(ValueError):
  """Raise this exception whenever a search must fail."""
  pass

### Question
Complete an implementation of the best-first search, encompassing A*, GBFS, or UCS.
      You can assume any heuristics are consistent.
      You should follow the psuedocode given in lecture closely.
      In particular, your implementation should prune redundant paths by remembering the reached states.

For reference, our solution is **55** line(s) of code.

In [4]:
def run_best_first_search(
    problem: PathCostProblem,
    get_priority: Callable[[Node], float],
    step_budget: int = 1000) -> Tuple[StateSeq, ActionSeq, CostSeq, int]:
  """A generic heuristic search implementation.

  Depending on `get_priority`, can implement A*, GBFS, or UCS.

  The `get_priority` function here should determine the order
  in which nodes are expanded. For example, if you want to
  use path cost as part of this determination, then the
  path cost (node.g) should appear inside of get_priority,
  rather than in this implementation of `run_best_first_search`.

  Important: for determinism (and to make sure our tests pass),
  please break ties using the state itself. For example,
  if you would've otherwise sorted by `get_priority(node)`, you
  should now sort by `(get_priority(node), node.state)`.

  Args:
    problem: a path cost problem.
    get_priority: a callable taking in a search Node and returns the priority
    step_budget: maximum number of `problem.step` before giving up.

  Returns:
    state_sequence: A list of states.
    action_sequence: A list of actions.
    cost_sequence: A list of costs.
    num_steps: number of taken `problem.step`s. Must be less than or equal to `step_budget`.

  Raises:
    error: SearchFailed, if no plan is found.
  """

  # holds only the states
  reached = {problem.initial.state}
  est_costs = {problem.initial.state: 0}
  agenda = [problem.initial] # heapq

  num_steps = 0
  while agenda and num_steps < step_budget:
    # stores by priority which is aware of the current best cost estimate
    curr_node = hq.heappop(agenda, lambda key: (get_priority(key.state, est_costs[key.state]), key.state))
    reached.add(curr_node.state)

    # late goal test for generic search
    if problem.goal_test(curr_node.state):
        return get_path(curr_node)

    # append neighbors if not already visited;
    # don't count them as visited until we've popped them
    for action in problem.actions(curr_node.state):
        neighbor_state = problem.step(curr_node, action)
        cost = problem.h(neighbor_state) + problem.step_cost(curr_node.state, action, neighbor_state)
        if cost <= est_costs:
            neighbor = Node()


SyntaxError: invalid syntax (3464977449.py, line 40)

### Tests

In [ ]:
# We will test this implementation more thoroughly with the
# specific heuristic search algorithms that follow
grid_problem = GridProblem()
get_priority_fn = lambda node: 0
result = run_best_first_search(grid_problem, get_priority_fn)
assert len(result) == 4


def best_first_search_test2():
  # We will test this implementation more thoroughly with the
  # specific heuristic search algorithms that follow
  grid_problem = GridProblem()
  get_priority_fn = lambda node: 0
  with count_method_calls(grid_problem, "step", "actions") as counters:
    state_sequence, action_sequence, cost_sequence, num_steps = run_best_first_search(
        grid_problem, get_priority_fn)
    assert (counters["step"].pop("total") == num_steps
           ), "Incorrect report of number of `problem.step`s"

  # Textbook implementation
  try:
    assert state_sequence == [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4),
                              (2, 4), (3, 4), (4, 4)]
    assert action_sequence == [
        'right', 'right', 'right', 'right', 'down', 'down', 'down', 'down'
    ]
    assert cost_sequence == [1.0, 8.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  # Alternative implementation that tracks best-cost-to-nodes
  except AssertionError:
    assert state_sequence == [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2),
                              (4, 2), (4, 3), (4, 4)]
    assert action_sequence == [
        'down', 'down', 'down', 'right', 'right', 'down', 'right', 'right'
    ]
    assert cost_sequence == [1.0, 1.0, 1.0, 1.0, 1.0, 2.0, 1.0, 1.0]
  assert num_steps <= 252

best_first_search_test2()


def best_first_search_test3():
  """If your results do not match the expected ones, make sure that you are
  tie-breaking as described in the docstring for `run_best_first_search`."""
  grid_problem = GridProblem()
  get_priority_fn = lambda node: node.g
  with count_method_calls(grid_problem, "step", "actions") as counters:
    state_sequence, action_sequence, cost_sequence, num_steps = run_best_first_search(
        grid_problem, get_priority_fn)
    assert (counters["step"].pop("total") == num_steps
           ), "Incorrect report of number of `problem.step`s"
  assert state_sequence == [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2),
                            (4, 2), (4, 3), (4, 4)]
  assert action_sequence == [
      'down', 'down', 'down', 'right', 'right', 'down', 'right', 'right'
  ]
  assert cost_sequence == [1.0, 1.0, 1.0, 1.0, 1.0, 2.0, 1.0, 1.0]
  assert num_steps <= 70

best_first_search_test3()

print('Tests passed.')

## Uniform Cost Search


### Question
Use your implementation of `run_best_first_search` to implement uniform cost search.

For reference, our solution is **3** line(s) of code.

In addition to all of the utilities defined at the top of the colab notebook, the following functions are available in this question environment: `run_best_first_search`. You may not need to use all of them.

In [ ]:
def run_uniform_cost_search(problem: PathCostProblem, step_budget: int = 1000):
  """Uniform-cost search.

  Use your implementation of `run_best_first_search`.
  """
  raise NotImplementedError("Implement me!")

### Tests

In [ ]:
def ucs_test1():
  # If your results do not match the expected ones, make sure that you are tiebreaking
  # as described in the docstring for `run_best_first_search`.
  grid_problem = GridProblem()
  state_sequence, action_sequence, cost_sequence, num_steps = run_uniform_cost_search(
      grid_problem)
  assert state_sequence == [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2),
                            (4, 2), (4, 3), (4, 4)]
  assert action_sequence == [
      'down', 'down', 'down', 'right', 'right', 'down', 'right', 'right'
  ]
  assert cost_sequence == [1.0, 1.0, 1.0, 1.0, 1.0, 2.0, 1.0, 1.0]
  assert num_steps <= 70

ucs_test1()

print('Tests passed.')

## A* Search


### Question
Use your implementation of `run_best_first_search` to implement A* search.

For reference, our solution is **3** line(s) of code.

In addition to all of the utilities defined at the top of the colab notebook, the following functions are available in this question environment: `run_best_first_search`. You may not need to use all of them.

In [ ]:
def run_astar_search(problem: PathCostProblem, step_budget: int = 1000):
  """A* search.

  Use your implementation of `run_best_first_search`.
  """
  raise NotImplementedError("Implement me!")

### Tests

In [ ]:
def astar_test1():
  """If your results do not match the expected ones, make sure that you are tiebreaking
  as described in the docstring for `run_best_first_search`."""
  grid_problem = GridProblem()
  state_sequence, action_sequence, cost_sequence, num_steps = run_astar_search(
      grid_problem)
  assert state_sequence == [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2),
                            (4, 2), (4, 3), (4, 4)]
  assert action_sequence == [
      'down', 'down', 'down', 'right', 'right', 'down', 'right', 'right'
  ]
  assert cost_sequence == [1.0, 1.0, 1.0, 1.0, 1.0, 2.0, 1.0, 1.0]
  assert num_steps <= 36

astar_test1()

print('Tests passed.')

## Greedy Best-First Search


### Question
Use your implementation of `run_best_first_search` to implement GBFS.

For reference, our solution is **3** line(s) of code.

In addition to all of the utilities defined at the top of the colab notebook, the following functions are available in this question environment: `run_best_first_search`. You may not need to use all of them.

In [ ]:
def run_greedy_best_first_search(problem: PathCostProblem,
                                 step_budget: int = 1000):
  """GBFS.

  Use your implementation of `run_best_first_search`.
  """
  raise NotImplementedError("Implement me!")

### Tests

In [ ]:
def gbfs_test1():
  """If your results do not match the expected ones, make sure that you are tiebreaking
  as described in the docstring for `run_best_first_search`."""
  problem = GridProblem()
  state_sequence, action_sequence, cost_sequence, num_steps = run_greedy_best_first_search(
      problem)
  assert state_sequence == [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4),
                            (2, 4), (3, 4), (4, 4)]
  assert action_sequence == [
      'right', 'right', 'right', 'right', 'down', 'down', 'down', 'down'
  ]
  assert abs(num_steps - 22) <= 1

gbfs_test1()

print('Tests passed.')